In [1]:
from diffusers import AutoPipelineForInpainting
import torch
import os, sys
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from tqdm import tqdm
import yaml
import tempfile
import cv2

# Add the project root directory to sys.path
project_root = os.path.abspath('..')  # Adjust this path if needed
sys.path.append(project_root)

from utils import load_images_and_combine, load_images
from utils import mask_on_image, dilate_mask, plot_grid

2024-08-28 17:51:08.303934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-08-28 17:51:08.318141: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-08-28 17:51:08.322532: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-08-28 17:51:08.335205: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-08-28 17:51:09.635083: W tensorflow/compiler/tf2

[08/28/24 17:51:15] WARNING  Your inference package version 0.12.1 is out of date! Please upgrade to ]8;id=802605;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py\__init__.py]8;;\:]8;id=402197;file:///usr/local/lib/python3.10/site-packages/inference/core/__init__.py#35\35]8;;\
                             version 0.16.2 of inference for the latest features and bug fixes by                  
                             running `pip install --upgrade inference`.                                            

In [2]:
init_model = 'lama' #@param ['lama', 'sd1.5', 'paint_by_example', 'ldm']
port = 4242
device = 'cuda' #@param ['cuda', 'cpu', 'mps']

cmd = f"iopaint start --model {init_model} --host 0.0.0.0 --port {port} --device {device}"
print(cmd)
# start subprocess using cmd
#subprocess.run(cmd, shell=True)

iopaint start --model lama --host 0.0.0.0 --port 4242 --device cuda


In [3]:
device2 = "cuda:2"
pipe = AutoPipelineForInpainting.from_pretrained("diffusers/stable-diffusion-xl-1.0-inpainting-0.1", torch_dtype=torch.float16, variant="fp16")
pipe = pipe.to(device2)
pipe.enable_attention_slicing() # Recommended if your computer has < 64 GB of RAM

generator = torch.Generator(device2).manual_seed(0)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

The config attributes {'decay': 0.9999, 'inv_gamma': 1.0, 'min_decay': 0.0, 'optimization_step': 37000, 'power': 0.6666666666666666, 'update_after_step': 0, 'use_ema_warmup': False} were passed to UNet2DConditionModel, but are not expected and will be ignored. Please verify your config.json configuration file.


In [4]:
# Load the YAML file into a dictionary
dataset_yaml_file = '/workspace/home/data/datasets.yaml'
with open(dataset_yaml_file, 'r') as file:
    datasets = yaml.safe_load(file)

In [5]:
# Load images
dataset_key ='ISIC2019'# 'Pascal_VOC07' #
dataset = datasets[dataset_key]
#images_confounder, images_no_confounder = load_images(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/raw", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])
#segmentation_masks_confounder, _ = load_images(dir_path=f"/workspace/home/data/{dataset['dataset_dir']}/seg", confounder_dir=dataset['confounder_dir'], no_confounder_dir=dataset['no_confounder_dir'])

segmentation_masks_confounder = load_images(dir_path=f"/workspace/datasets/ISIC19/segmentation", ignore_copy=True)


output_dir = f"/workspace/home/outputfiles/{dataset['dataset_dir']}/inpainting/iopaint/{init_model}"

output_dir_confounder = os.path.join(output_dir, dataset['confounder_dir'])
#output_dir_no_confounder = os.path.join(output_dir, dataset['no_confounder_dir'])

# Create output directories if they don't exist
os.makedirs(output_dir_confounder, exist_ok=True)
#os.makedirs(output_dir_no_confounder, exist_ok=True)

In [1]:
prompt = "part of skin in homogenous skin color"
# "part of skin"
#"remove the confounder"
# ""
#"remove colorful patch"
#"lesion on skin"

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True

start = 1889 - 557 - 64

#for i, (image_dict, mask_dict) in enumerate(tqdm(zip(images_confounder, segmentation_masks_confounder))):
for i, mask_dict in enumerate(tqdm(segmentation_masks_confounder)):
  if i < start:
    continue
  mask = mask_dict['image']
  meta_mask = mask_dict['meta']
  mask_url = meta_mask['file_path']

  img_url = mask_url.replace('segmentation', 'raw')

  # Load the image
  image = Image.open(img_url).convert('RGB')
  original_img_size = image.size

  # Dilate the mask
  dilated_mask = dilate_mask(Image.open(mask_url), kernel_size=5, iterations=5)
  resized_img = image.resize(dilated_mask.size)


  with tempfile.TemporaryDirectory() as tmpdirname:

    # Define the path for the temporary mask file
    mask_path = os.path.join(tmpdirname, 'mask.jpg')
    img_url = os.path.join(tmpdirname, meta_mask['file_name'])
    # Save the dilated mask to the temporary file
    cv2.imwrite(mask_path, np.array(dilated_mask))
    resized_img_array = np.array(resized_img)
    cv2.imwrite(img_url, cv2.cvtColor(resized_img_array, cv2.COLOR_RGB2BGR))

    !iopaint run --model={init_model} --device={device} --image={img_url} --mask={mask_path} --output={output_dir_confounder}

    # path for second iteration
    img_url = os.path.join(output_dir_confounder, meta_mask['file_name'].replace('.jpg', '.png'))
    output_dir_confounder2 = f"/workspace/home/data/{dataset['dataset_dir']}/inpainted/"


    #!iopaint run --model="diffusers/stable-diffusion-xl-1.0-inpainting-0.1" --device={device} --image={img_url} --mask={mask_path} --output={output_dir_confounder2}

    image = Image.open(img_url)
    mask = dilated_mask if isinstance(dilated_mask, Image.Image) else np.array(dilated_mask)
    image = image.resize((1024, 1024))
    mask = mask.resize((1024, 1024))
    #Run the diffuser
    result = pipe(
                prompt=prompt,
                image=image,
                mask_image=mask,
                guidance_scale=8.0,
                num_inference_steps=25,  # steps between 15 and 30 work well for us
                strength=0.99,  # make sure to use `strength` below 1.0
                generator=generator,
                #padding_mask_crop=32,
                ).images[0]

    result = result.resize(original_img_size)

    #Save the inpainted image
    result.save(f"{output_dir_confounder2}/{meta_mask['file_name']}")

    print(f"Saved {output_dir_confounder2}/{meta_mask['file_name']}")


# for image_dict in images_no_confounder:
#   image = image_dict['image']
#   meta_img = image_dict['meta']
#   img_url = meta_img['file_path']

#   # save image as PIL image
#   img = Image.open(img_url)
#   img.save(f"{output_dir_no_confounder}/{meta_img['file_name']}")


NameError: name 'tqdm' is not defined